# Notebook 03 — KMeans Clustering
**Project:** Credit Card Customer Profiling & Segmentation  
**Goal:** Find the optimal number of clusters using Elbow + Silhouette analysis, fit KMeans, and visualise segments in PCA space.

---
| Step | Action |
|------|--------|
| 1 | Load scaled data |
| 2 | Elbow Method — Inertia vs K |
| 3 | Silhouette Score analysis |
| 4 | Fit KMeans with optimal K |
| 5 | PCA 2D visualisation |
| 6 | Cluster size distribution |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

from src.data_loader import load_raw_data
from src.preprocessing import preprocess
from src.config import RANDOM_STATE, K_RANGE, N_CLUSTERS, MODELS_DIR

df_scaled, df_unscaled = preprocess(load_raw_data(), save=False)
print('Scaled shape:', df_scaled.shape)

## 1. Find Optimal K

In [ ]:
inertias, silhouettes = [], []

print(f'{"K":>3} | {"Inertia":>12} | {"Silhouette":>10}')
print('-' * 32)
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(df_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(df_scaled, labels)
    silhouettes.append(sil)
    print(f'{k:>3} | {km.inertia_:>12,.0f} | {sil:>10.4f}')

In [ ]:
palette = ['#2196F3','#F44336','#4CAF50','#FF9800']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_RANGE), inertias, marker='o', color='#2196F3', lw=2.5, ms=8)
axes[0].fill_between(list(K_RANGE), inertias, alpha=0.07, color='#2196F3')
axes[0].axvline(N_CLUSTERS, color='#F44336', ls='--', lw=2, label=f'Chosen K={N_CLUSTERS}')
axes[0].set_title('Elbow Method', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].legend()

axes[1].plot(list(K_RANGE), silhouettes, marker='s', color='#4CAF50', lw=2.5, ms=8)
axes[1].fill_between(list(K_RANGE), silhouettes, alpha=0.07, color='#4CAF50')
axes[1].axvline(N_CLUSTERS, color='#F44336', ls='--', lw=2, label=f'Chosen K={N_CLUSTERS}')
axes[1].set_title('Silhouette Score', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

plt.suptitle(f'Optimal K Selection — Chosen K={N_CLUSTERS}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Fit KMeans

In [ ]:
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
km_labels = kmeans.fit_predict(df_scaled)
sil_final = silhouette_score(df_scaled, km_labels)

joblib.dump(kmeans, MODELS_DIR / 'kmeans.pkl')
print(f'KMeans K={N_CLUSTERS} | Silhouette={sil_final:.4f} | Inertia={kmeans.inertia_:,.0f}')
print('\nCluster sizes:')
for k, n in zip(*np.unique(km_labels, return_counts=True)):
    print(f'  Cluster {k}: {n:,} customers ({n/len(km_labels)*100:.1f}%)')
print('\nModel saved: models/kmeans.pkl')

## 3. PCA Visualisation

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
comps = pca.fit_transform(df_scaled)
explained = pca.explained_variance_ratio_.sum() * 100
joblib.dump(pca, MODELS_DIR / 'pca.pkl')
print(f'PC1: {pca.explained_variance_ratio_[0]*100:.1f}%  PC2: {pca.explained_variance_ratio_[1]*100:.1f}%  Total: {explained:.1f}%')

fig, ax = plt.subplots(figsize=(9, 6))
for cluster in np.unique(km_labels):
    mask = km_labels == cluster
    ax.scatter(comps[mask, 0], comps[mask, 1],
               label=f'Cluster {cluster}', alpha=0.45, s=14,
               color=palette[cluster % len(palette)])
ax.set_title(f'KMeans Segments — PCA 2D ({explained:.1f}% variance)', fontsize=13, fontweight='bold')
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.legend(markerscale=3, title='Segment')
plt.tight_layout()
plt.show()

## 4. Cluster Distribution

In [ ]:
unique, counts = np.unique(km_labels, return_counts=True)
pcts = counts / counts.sum() * 100
labels_str = [f'Cluster {k}\n({p:.1f}%)' for k, p in zip(unique, pcts)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
bars = axes[0].bar(labels_str, counts, color=palette[:len(unique)], edgecolor='white', alpha=0.88)
for bar, n in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{n:,}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Customer Count per Segment', fontweight='bold')
axes[0].set_ylabel('Customers')

axes[1].pie(counts, labels=labels_str, colors=palette[:len(unique)],
            autopct='%1.1f%%', startangle=140,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[1].set_title('Segment Distribution', fontweight='bold')
plt.suptitle('Customer Distribution Across Segments', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Results Summary

| Metric | Value |
|---|---|
| Algorithm | KMeans |
| Optimal K | 4 |
| Silhouette Score | 0.1758 |
| Inertia (WCSS) | 152,856 |
| PCA variance (2D) | 40.5% |

> **Next:** `04_Hierarchical_Clustering.ipynb` — compare with Agglomerative Hierarchical Clustering.